# Setup do ambiente
Executa o `setup.sql` substituindo os catálogos pelos parâmetros do target (dev ou prd).

In [ ]:
import re

for nome, padrao in [("catalogo_bronze", "dev_bronze"), ("catalogo_silver", "dev_silver"),
                     ("catalogo_gold", "dev_gold"), ("setup_sql_path", "")]:
    dbutils.widgets.text(nome, padrao)

parametros = {nome: dbutils.widgets.get(nome) for nome in ["catalogo_bronze", "catalogo_silver", "catalogo_gold"]}
for nome, valor in parametros.items():
    if not re.fullmatch(r"[a-z][a-z0-9_]{0,62}", valor):
        raise ValueError(f"Parâmetro {nome} inválido: {valor!r}")

In [ ]:
with open(dbutils.widgets.get("setup_sql_path"), encoding="utf-8") as arquivo:
    script = arquivo.read()

for nome, valor in parametros.items():
    script = script.replace("${" + nome + "}", valor)

linhas = [linha for linha in script.splitlines() if not linha.strip().startswith("--")]
comandos = [c.strip() for c in "\n".join(linhas).split(";") if c.strip()]
for comando in comandos:
    print(comando.splitlines()[0])
    spark.sql(comando)
print(f"Setup concluído: {len(comandos)} comandos executados.")